# Notebook 4 — One Transfer Step

**Course: Cross-Corridor Capacity Analysis with pandapower (Svedala grid)**

Before automating the sweep, we need a clean primitive: **apply one transfer**
**increment and observe the result**.

## Learning objectives

- Apply a generation shift `+ΔP` from source to sink using GSK.
- Run the power flow on the new dispatch.
- Read off the new corridor flow and the most-stressed monitored elements.
- Restore the base dispatch cleanly using `copy.deepcopy`.

## 4.1  Load everything

In [ ]:
import json, copy
import pandapower as pp
import pandas as pd
import numpy as np

net = pp.from_json('data/svedala_base.json')
pp.runpp(net)

with open('data/corridor.json') as f:
    corridor = json.load(f)

GSK_A = pd.Series({int(k): float(v) for k, v in corridor['gsk_a'].items()})
GSK_B = pd.Series({int(k): float(v) for k, v in corridor['gsk_b'].items()})
LINE_DIR  = [tuple(x) for x in corridor['corridor_lines']]
TRAFO_DIR = [tuple(x) for x in corridor['corridor_trafos']]
P0        = corridor['base_flow_a_to_b']
ZONE_SOURCE = corridor['zone_source']
ZONE_SINK   = corridor['zone_sink']

print(f'Base flow {ZONE_SOURCE} → {ZONE_SINK}: {P0:+.2f} MW')

## 4.2  The transfer-step primitive

The function takes a fresh deep-copy of the base net, modifies generator
setpoints in-place, runs the power flow, and reports.

Generator setpoints are clipped to `[min_p_mw, max_p_mw]` to model saturation:
if a generator can't take its full GSK share, the rest spills onto the slack.

In [ ]:
def corridor_flow(net, line_dir, trafo_dir):
    p = 0.0
    for idx, d in line_dir:
        p += d * net.res_line.at[idx, 'p_from_mw']
    for idx, d in trafo_dir:
        p += d * net.res_trafo.at[idx, 'p_hv_mw']
    return p

def apply_transfer_step(base_net, delta_mw, gsk_a, gsk_b,
                        line_dir, trafo_dir,
                        v_min=0.95, v_max=1.05,
                        line_limit=100.0, trafo_limit=100.0):
    """Apply a one-shot generation shift of delta_mw and analyse."""
    net = copy.deepcopy(base_net)

    # Saturating shift
    for g, k in gsk_a.items():
        new_p = net.gen.at[g, 'p_mw'] + k * delta_mw
        if not np.isnan(net.gen.at[g, 'max_p_mw']):
            new_p = min(new_p, net.gen.at[g, 'max_p_mw'])
        if not np.isnan(net.gen.at[g, 'min_p_mw']):
            new_p = max(new_p, net.gen.at[g, 'min_p_mw'])
        net.gen.at[g, 'p_mw'] = new_p
    for g, k in gsk_b.items():
        new_p = net.gen.at[g, 'p_mw'] - k * delta_mw
        if not np.isnan(net.gen.at[g, 'max_p_mw']):
            new_p = min(new_p, net.gen.at[g, 'max_p_mw'])
        if not np.isnan(net.gen.at[g, 'min_p_mw']):
            new_p = max(new_p, net.gen.at[g, 'min_p_mw'])
        net.gen.at[g, 'p_mw'] = new_p

    try:
        pp.runpp(net)
    except pp.LoadflowNotConverged:
        return {'delta_request_mw': delta_mw, 'converged': False,
                'corridor_flow': None, 'max_line_loading': None,
                'min_voltage': None, 'max_voltage': None}

    return {
        'delta_request_mw':  delta_mw,
        'converged':         True,
        'corridor_flow':     corridor_flow(net, line_dir, trafo_dir),
        'max_line_loading':  float(net.res_line.loading_percent.max()),
        'max_trafo_loading': float(net.res_trafo.loading_percent.max())
                              if len(net.res_trafo) else 0.0,
        'min_voltage':       float(net.res_bus.vm_pu.min()),
        'max_voltage':       float(net.res_bus.vm_pu.max()),
    }

## 4.3  Try a single step

In [ ]:
result = apply_transfer_step(net, 200.0, GSK_A, GSK_B, LINE_DIR, TRAFO_DIR)
for k, v in result.items():
    print(f'  {k:<20} {v}')

Notice that the *achieved* corridor flow is not exactly `P0 + 200` MW — losses
change with flow patterns and the slack absorbs the difference, plus generators
may saturate. That's why we always **measure** the corridor flow rather than
trusting the requested ΔP.

## 4.4  Sanity sweep — a handful of small steps

In [ ]:
rows = []
for delta in [0, 50, 100, 200, 300, 500, 700, 1000]:
    rows.append(apply_transfer_step(net, delta, GSK_A, GSK_B,
                                    LINE_DIR, TRAFO_DIR))
df = pd.DataFrame(rows)
df.round(3)

In [ ]:
import matplotlib.pyplot as plt
ok = df[df.converged]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ok.corridor_flow, ok.max_line_loading, 'o-', label='max line loading')
ax.axhline(100, color='red', linestyle='--', label='100 % limit')
ax.set_xlabel(f'Corridor flow {ZONE_SOURCE} → {ZONE_SINK} [MW]')
ax.set_ylabel('Max line loading [%]')
ax.set_title('Pre-N-1 capacity probe (single-step)')
ax.grid(True, alpha=0.4); ax.legend()
plt.tight_layout(); plt.show()

## 4.5  Reverse direction (sink → source)

Capacity is direction-dependent. Test by passing negative `delta_mw`.

In [ ]:
for delta in [-100, -300, -500]:
    r = apply_transfer_step(net, delta, GSK_A, GSK_B, LINE_DIR, TRAFO_DIR)
    if r['converged']:
        print(f'ΔP = {delta:+5} MW → corridor flow {r["corridor_flow"]:+8.2f} MW, '
              f'max line loading {r["max_line_loading"]:5.1f} %')
    else:
        print(f'ΔP = {delta:+5} MW → did not converge')

## 4.6  Exercises

1. Find the **smallest** positive `ΔP` (resolution 10 MW, then bisect to 1 MW)
   that produces a *line* violation. Report the limiting line.
2. Find the **smallest** positive `ΔP` that produces a *voltage* violation.
   Which limit binds first — thermal or voltage? In Sweden, long
   NORR → MITT transfers historically bind on voltage. Does Svedala behave
   that way?
3. Replace Pmax-proportional GSK in the source zone with **headroom-
   proportional** GSK (each gen contributes proportionally to `max_p_mw - p_mw`)
   and rerun the small sweep. Does the corridor flow vs ΔP relation change?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


---

✅ **Checkpoint reached.** Transfer-step primitive working.

Continue to [Notebook 5 — Iterative Capacity Probing](05_capacity_sweep.ipynb).